# Folium Mapping Tutorial
This notebook walks through the workflow for building an interactive compaction map from CKAN-discovered Upstream station resources. The Folium markers point to a reusable popup page that fetches each station's Upstream measurement JSON directly when the popup opens.


In [ ]:
import json
import mimetypes
import os
from getpass import getpass
from pathlib import Path
from typing import Any
from urllib.parse import quote, urlencode

import pandas as pd
import numpy as np
import geopandas as gpd

import plotly.express as px
from plotly import graph_objects as go
from plotly.subplots import make_subplots

import folium
import branca
from branca.element import MacroElement
from jinja2 import Template
import requests

def get_tapis_token(username: str, password: str, *, tapis_url: str) -> str:
    response = requests.post(
        tapis_url,
        data={
            'username': username,
            'password': password,
            'grant_type': 'password',
        },
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    return payload['result']['access_token']['access_token']

class CKANClient:
    def __init__(self, ckan_url: str, auth_header: str):
        self.ckan_url = ckan_url.rstrip('/')
        self.headers = {'Authorization': auth_header}

    def action(
        self,
        action: str,
        *,
        method: str = 'GET',
        params: dict[str, Any] | None = None,
        json_payload: dict[str, Any] | None = None,
        data: dict[str, Any] | None = None,
        files: dict[str, Any] | None = None,
    ) -> Any:
        response = requests.request(
            method,
            f'{self.ckan_url}/api/3/action/{action}',
            headers=self.headers,
            params=params,
            json=json_payload,
            data=data,
            files=files,
            timeout=120,
        )
        response.raise_for_status()
        payload = response.json()
        if not payload.get('success'):
            raise RuntimeError(payload.get('error') or payload)
        return payload['result']

    def package_show(self, dataset_id: str) -> dict[str, Any]:
        return self.action('package_show', params={'id': dataset_id})

    def resource_create(
        self,
        package_id: str,
        file_path: Path,
        *,
        name: str,
        description: str,
        format_name: str,
    ) -> dict[str, Any]:
        content_type = mimetypes.guess_type(file_path.name)[0] or 'application/octet-stream'
        with file_path.open('rb') as fh:
            return self.action(
                'resource_create',
                method='POST',
                data={
                    'package_id': package_id,
                    'name': name,
                    'description': description,
                    'format': format_name,
                },
                files={'upload': (file_path.name, fh, content_type)},
            )

    def resource_update(
        self,
        resource_id: str,
        package_id: str,
        file_path: Path,
        *,
        name: str,
        description: str,
        format_name: str,
    ) -> dict[str, Any]:
        content_type = mimetypes.guess_type(file_path.name)[0] or 'application/octet-stream'
        with file_path.open('rb') as fh:
            return self.action(
                'resource_update',
                method='POST',
                data={
                    'id': resource_id,
                    'package_id': package_id,
                    'name': name,
                    'description': description,
                    'format': format_name,
                },
                files={'upload': (file_path.name, fh, content_type)},
            )

    def resource_view_list(self, resource_id: str) -> list[dict[str, Any]]:
        return self.action('resource_view_list', params={'id': resource_id})

    def resource_view_create(self, payload: dict[str, Any]) -> dict[str, Any]:
        return self.action('resource_view_create', method='POST', json_payload=payload)

    def resource_view_update(self, payload: dict[str, Any]) -> dict[str, Any]:
        return self.action('resource_view_update', method='POST', json_payload=payload)


## Load Data
Start by importing the libraries used in the workflow. `pandas` and `geopandas` handle tabular and spatial data, `plotly` creates the popup charts, and `folium` builds the interactive web map.


### Read County Boundaries
Load the Texas county GeoJSON into a GeoDataFrame so selected counties can be drawn as overlays on the Folium map.


#### Find Data From Ckan

In [ ]:
txgeojson='https://ckan.tacc.utexas.edu/dataset/cd3deceb-7102-44b1-a83b-35da7c8f6855/resource/204f8874-0db4-4e81-95e3-e695f4056bdc/download/texas_county_boundaries_detailed.geojson'
tx_gdf_county = gpd.read_file(txgeojson)

### Check the Spatial Layer
Filter to one county as a quick validation step. This is useful for confirming the boundary file loaded correctly and that the county names match what you expect to use later.


In [ ]:
tx_gdf_county[tx_gdf_county['CNTY_NM']=='Fort Bend']

### Discover Upstream Station Resources from CKAN
Query CKAN for the Houston-area extensometer campaign packages, then build a station table from the package metadata and each package's Upstream measurement resource URL.


In [ ]:
CKAN_URL = 'https://ckan.tacc.utexas.edu'
CKAN_SEARCH_URL = f'{CKAN_URL}/api/3/action/package_search'
CAMPAIGN_TAG = 'houston-area-extensometer-compaction-campaign'


def extract_extras(package):
    return {extra['key']: extra['value'] for extra in package.get('extras', [])}


def extract_value(text, prefix, suffix=None):
    if prefix:
        if prefix not in text:
            return None
        value = text.split(prefix, 1)[1]
    else:
        value = text
    if suffix and suffix in value:
        value = value.split(suffix, 1)[0]
    return value.strip()


def spatial_center(spatial_text):
    spatial = json.loads(spatial_text)
    coordinates = spatial['coordinates']
    if spatial.get('type') == 'Point':
        return coordinates

    if spatial.get('type') == 'Polygon':
        ring = coordinates[0]
        ring_points = ring[:-1] or ring
        longitudes = [point[0] for point in ring_points]
        latitudes = [point[1] for point in ring_points]
        return sum(longitudes) / len(longitudes), sum(latitudes) / len(latitudes)

    raise ValueError(f"Unsupported spatial geometry type: {spatial.get('type')}")


def measurement_resource(package):
    for resource in package.get('resources', []):
        name = resource.get('name', '')
        if resource.get('format') == 'GeoJSON' and name.endswith('-measurements'):
            return resource
    raise ValueError(f"No measurement resource found for {package['name']}")


def sensor_id_from_measurement_url(url):
    if '/sensors/' not in url:
        return None
    return url.split('/sensors/', 1)[1].split('/', 1)[0]


def discover_station_packages():
    params = {
        'q': f'tags:{CAMPAIGN_TAG} AND tags:upstream',
        'rows': 100,
    }
    response = requests.get(CKAN_SEARCH_URL, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if not payload.get('success'):
        raise RuntimeError(payload.get('error') or payload)
    packages = payload['result']['results']
    return sorted(packages, key=lambda package: int(extract_extras(package)['station_id']))


def build_sites_dataframe(packages):
    rows = []
    for package in packages:
        extras = extract_extras(package)
        resource = measurement_resource(package)
        measurement_url = resource['url']
        longitude, latitude = spatial_center(package['spatial'])
        notes = package.get('notes', '')
        general_name = extract_value(notes, '', ' extensometer station') or extras['station_name']
        station_id = int(extras['station_id'])
        campaign_id = int(extras['campaign_id'])

        rows.append({
            'SITE_NO': extract_value(notes, 'source_site_no='),
            'STATION_NM': extras['station_name'],
            'COMPACTION_INTERVAL': extract_value(notes, 'interval=', ';'),
            'ANCHOR_DEPTH': extract_value(notes, 'anchor_depth=', ' ft'),
            'DEC_LONG_VA': longitude,
            'DEC_LAT_VA': latitude,
            'GENERAL_NM': general_name,
            'name_condensed': general_name.replace(' ', ''),
            'campaign_id': campaign_id,
            'station_id': station_id,
            'sensor_id': sensor_id_from_measurement_url(measurement_url),
            'measurement_resource_id': resource['id'],
            'measurement_url': measurement_url,
            'dataset_name': package['name'],
            'temporal_coverage_start': package.get('temporal_coverage_start'),
            'temporal_coverage_end': package.get('temporal_coverage_end'),
        })
    return pd.DataFrame(rows)


station_packages = discover_station_packages()
sites_df = build_sites_dataframe(station_packages)


### Inspect the Site Table
Preview the CKAN-derived station table and confirm each site has a browser-accessible Upstream measurement URL.


In [ ]:
sites_df[['station_id', 'sensor_id', 'GENERAL_NM', 'STATION_NM', 'measurement_url']].head()


### Load Compaction Measurements for Notebook Preview
Fetch each Upstream measurement endpoint once so the notebook can preview the same data that the popup HTML will fetch in the browser.


In [ ]:
def fetch_measurement_items(measurement_url, *, page_size=1000):
    page = 1
    items = []
    while True:
        response = requests.get(measurement_url, params={'limit': page_size, 'page': page}, timeout=60)
        response.raise_for_status()
        payload = response.json()
        page_items = payload.get('items', [])
        items.extend(page_items)

        total_pages = payload.get('pages') or page
        if page >= total_pages or len(page_items) < page_size:
            break
        page += 1
    return items


def load_measurements_from_site(site_row):
    items = fetch_measurement_items(site_row['measurement_url'])
    if not items:
        return pd.DataFrame(columns=['DATE', 'CUMULATIVE_COMPACTION', 'site', 'data_version'])

    measurements = pd.DataFrame(items)
    measurements['DATE'] = pd.to_datetime(measurements['collectiontime'])
    measurements['CUMULATIVE_COMPACTION'] = pd.to_numeric(measurements['value'], errors='coerce')
    measurements['site'] = site_row['name_condensed']
    measurements['data_version'] = measurements['DATE'].dt.year.max()
    return measurements[['DATE', 'CUMULATIVE_COMPACTION', 'site', 'data_version']]


measurement_frames = [load_measurements_from_site(row) for _, row in sites_df.iterrows()]
df = pd.concat(measurement_frames, ignore_index=True).sort_values(['site', 'DATE']).reset_index(drop=True)


### Clean the Date Column
The Upstream JSON already provides ISO-like timestamps. Convert them to datetimes and keep the table sorted for plotting.


In [ ]:
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['site', 'DATE']).reset_index(drop=True)


### Preview the Cleaned Measurements
Check the cleaned table to confirm the dates and compaction values are ready for plotting.


In [ ]:
df.head()


## Optional CKAN Registration Checkpoint

Before moving into popup generation and map assembly, this is still a good point to review the CKAN station packages and the Upstream measurement resources discovered from them. The popup HTML generated below will use those measurement URLs directly rather than embedding a static copy of the data.


### Recommended Registration Targets

Treat the Upstream measurement endpoints as the source data product, then upload the generated HTML artifacts as companion visualization resources on the same CKAN dataset. That keeps the measurements discoverable while still publishing the interactive outputs that this notebook depends on.


## Build and Preview a Site Chart
Before automating the map popups, isolate one site and make a simple Plotly figure. This lets you confirm the compaction time series looks right for a single location first.


### Subset a Single Site
Start with the `TexasCity` site so you can test the plotting workflow on one location before looping through the entire dataset.


In [ ]:
tc = df[df.site=='TexasCity']

### Inspect the Site-Specific Records
Display a couple of rows from the filtered dataset to confirm the subset contains the expected measurements.


In [ ]:
tc.head(2)

### Create a Quick Plotly Figure
Build a simple scatter plot of cumulative compaction over time. This is a first visual check before defining a reusable chart function.


In [ ]:
fig = px.scatter(tc, x="DATE", y="CUMULATIVE_COMPACTION")

### Render the Preview Figure
Display the Plotly figure inline to confirm the data and axes look reasonable.


In [ ]:
fig

### Wrap the Plot Logic in a Function
Define a helper function that takes a site name, pulls the matching metadata and measurements, and returns a formatted Plotly figure. This makes it easy to generate one popup chart per site later.


In [ ]:
def make_popup_figure(site, sites_df, compaction_df):
    sitename = sites_df.loc[sites_df['name_condensed'] == site, 'GENERAL_NM'].item()
    plot_data = compaction_df[compaction_df.site == site]
    title = 'Cumulative Compaction at ' + sitename + ' extensometer'
    fig = px.line(plot_data, x="DATE", y="CUMULATIVE_COMPACTION",
                 title=title,
                 labels={
                     "DATE": 'Measurement date',
                     "CUMULATIVE_COMPACTION": 'Cumulative Compaction (feet)'
                 }
                )
    fig.update_layout(
        title_font_weight='bold'
    )
    return fig

### Test the Reusable Figure Function
Call the helper once for `TexasCity` to verify it produces the expected popup chart before running it in a loop.


In [ ]:
make_popup_figure('TexasCity',sites_df, df)

## Build a Reusable Popup Page
Each map marker will open the same small HTML page. The marker passes its Upstream measurement URL in the iframe query string, and the browser fetches the JSON directly when the popup opens.


### Prepare Output Storage
Create a folder for the popup template and final Folium map if it does not already exist.


In [ ]:
output_dir = Path('folium_html')
output_dir.mkdir(exist_ok=True)

popup_template_path = output_dir / 'popup.html'
popup_template_path


### Write the Browser-Fetched Popup Template
The template contains only rendering code. It reads the `url`, `site`, and `title` query parameters, fetches the Upstream JSON endpoint, and renders the Plotly chart in the browser.


In [ ]:
popup_template = '''<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Compaction popup</title>
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
  <style>
    html, body {
      height: 100%;
      margin: 0;
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: #24313d;
      background: #ffffff;
    }
    #status {
      box-sizing: border-box;
      min-height: 28px;
      padding: 8px 10px 0;
      font-size: 12px;
      color: #53616f;
    }
    #chart {
      width: 100%;
      height: calc(100% - 28px);
      min-height: 310px;
    }
    .error {
      color: #9b1c1c;
    }
  </style>
</head>
<body>
  <div id="status">Loading measurements...</div>
  <div id="chart"></div>
  <script>
    const params = new URLSearchParams(window.location.search);
    const measurementUrl = params.get("url");
    const site = params.get("site") || "site";
    const title = params.get("title") || `Cumulative Compaction at ${site}`;
    const statusEl = document.getElementById("status");

    function fail(message) {
      statusEl.textContent = message;
      statusEl.className = "error";
    }

    function validMeasurementUrl(url) {
      return url && url.startsWith("https://upstreamapi.pods.portals.tapis.io/api/v1/");
    }

    async function fetchMeasurementItems(url) {
      const allItems = [];
      let page = 1;
      let totalPages = 1;

      while (page <= totalPages) {
        const pageUrl = new URL(url);
        pageUrl.searchParams.set("limit", "1000");
        pageUrl.searchParams.set("page", String(page));

        const response = await fetch(pageUrl.toString());
        if (!response.ok) {
          throw new Error(`HTTP ${response.status}`);
        }

        const payload = await response.json();
        const pageItems = Array.isArray(payload.items) ? payload.items : [];
        allItems.push(...pageItems);
        totalPages = Number(payload.pages || page);

        if (!pageItems.length) {
          break;
        }
        page += 1;
      }

      return allItems;
    }

    async function render() {
      if (!validMeasurementUrl(measurementUrl)) {
        fail("Missing or unsupported Upstream measurement URL.");
        return;
      }

      try {
        const items = await fetchMeasurementItems(measurementUrl);
        if (!items.length) {
          fail("No measurements returned from Upstream.");
          return;
        }

        items.sort((a, b) => new Date(a.collectiontime) - new Date(b.collectiontime));
        const x = items.map((item) => item.collectiontime);
        const y = items.map((item) => Number(item.value));
        const latest = new Date(x[x.length - 1]);

        statusEl.textContent = `${items.length} measurements fetched from Upstream. Latest: ${latest.toISOString().slice(0, 10)}`;
        statusEl.className = "";

        Plotly.newPlot("chart", [{
          x,
          y,
          type: "scatter",
          mode: "lines+markers",
          line: { color: "#2563a8", width: 2 },
          marker: { color: "#2563a8", size: 5 },
          hovertemplate: "%{x|%Y-%m-%d}<br>%{y:.3f} ft<extra></extra>"
        }], {
          title: { text: title, font: { size: 15 } },
          margin: { l: 56, r: 18, t: 54, b: 46 },
          xaxis: { title: "Measurement date" },
          yaxis: { title: "Cumulative Compaction (feet)" },
          template: "plotly_white",
          hovermode: "x unified"
        }, {
          responsive: true,
          displaylogo: false
        });
      } catch (error) {
        fail(`Could not load Upstream measurements: ${error.message}`);
      }
    }

    render();
  </script>
</body>
</html>
'''

popup_template_path.write_text(popup_template, encoding='utf-8')


### Build a Popup Template Upload Record
Create a one-row table for the reusable popup file that will be uploaded to CKAN.


In [ ]:
popup_template_df = pd.DataFrame([{
    'html_file': popup_template_path.name,
    'html_path': popup_template_path.as_posix(),
    'size_kb': round(popup_template_path.stat().st_size / 1024, 1),
}])
popup_template_df


### Attach Popup Parameters to the Site Metadata
Each marker keeps its own Upstream measurement URL. After `popup.html` is uploaded to CKAN, the map will combine the template URL with these site-specific query parameters.


In [ ]:
def build_popup_url(template_url, site_row):
    params = {
        'url': site_row['measurement_url'],
        'site': site_row['name_condensed'],
        'title': f"Cumulative Compaction at {site_row['GENERAL_NM']} extensometer",
    }
    return f"{template_url}?{urlencode(params)}"


sites_df[['name_condensed', 'measurement_url']].head()


### Verify the Popup Template
Preview the local template file size. The final popup URLs are created after the template is uploaded to CKAN.


In [ ]:
popup_template_df


## Publish Popup HTML to CKAN
Use the notebook's local CKAN helpers to authenticate with TACC, upload or update the reusable popup template, and build one browser-fetched popup URL per site.


### Authenticate and Upload the Popup Template
The helper below exchanges TACC credentials for a bearer token, connects to CKAN, and creates or updates `popup.html` by resource name.


In [ ]:
CKAN_URL = 'https://ckan.tacc.utexas.edu'
TAPIS_TOKEN_URL = 'https://portals.tapis.io/v3/oauth2/tokens'
CKAN_HTML_DATASET_ID = os.environ.get('CKAN_HTML_DATASET_ID', '961271e4-c3e3-48f8-a9c1-c2829ccf0cf1')
MAX_UPLOAD_MB_WARNING = float(os.environ.get('CKAN_MAX_UPLOAD_MB_WARNING', '5'))


def build_ckan_client():
    username = os.environ.get('TACC_USERNAME') or input('TACC username: ').strip()
    password = os.environ.get('TACC_PASSWORD') or getpass('TACC password: ')
    auth_header = f"Bearer {get_tapis_token(username, password, tapis_url=TAPIS_TOKEN_URL)}"
    return CKANClient(CKAN_URL, auth_header)


def resource_download_url(dataset, resource):
    filename = quote(resource.get('name') or Path(resource.get('url', 'resource')).name)
    return f"{CKAN_URL}/dataset/{dataset['name']}/resource/{resource['id']}/download/{filename}"


def file_size_mb(file_path):
    return Path(file_path).stat().st_size / (1024 * 1024)


def upsert_resource_by_name(client, dataset, file_path, *, name=None, description='', format_name='HTML'):
    file_path = Path(file_path)
    resource_name = name or file_path.name
    size_mb = file_size_mb(file_path)
    if size_mb > MAX_UPLOAD_MB_WARNING:
        raise ValueError(
            f"{file_path.name} is {size_mb:.2f} MB before upload. Regenerate with smaller HTML or raise the warning threshold if the CKAN server allows it."
        )
    existing = next((resource for resource in dataset.get('resources', []) if resource.get('name') == resource_name), None)
    if existing:
        resource = client.resource_update(
            existing['id'],
            dataset['id'],
            file_path,
            name=resource_name,
            description=description,
            format_name=format_name,
        )
        dataset['resources'] = [resource if item.get('id') == resource['id'] else item for item in dataset.get('resources', [])]
    else:
        resource = client.resource_create(
            dataset['id'],
            file_path,
            name=resource_name,
            description=description,
            format_name=format_name,
        )
        dataset.setdefault('resources', []).append(resource)
    return resource


def upsert_webpage_view(client, resource_id, *, title, description=''):
    existing_views = client.resource_view_list(resource_id)
    existing = next((view for view in existing_views if view.get('view_type') == 'webpage_view'), None)
    payload = {
        'resource_id': resource_id,
        'title': title,
        'description': description,
        'view_type': 'webpage_view',
    }
    if existing:
        payload['id'] = existing['id']
        return client.resource_view_update(payload)
    return client.resource_view_create(payload)


upload_size_df = popup_template_df.assign(size_mb=popup_template_df['html_path'].map(lambda p: round(file_size_mb(p), 3)))
upload_size_df

ckan_client = build_ckan_client()
html_dataset = ckan_client.package_show(CKAN_HTML_DATASET_ID)

popup_template_resource = upsert_resource_by_name(
    ckan_client,
    html_dataset,
    popup_template_path,
    name=popup_template_path.name,
    description='Reusable Plotly popup shell that fetches Upstream measurement JSON in the browser.',
    format_name='HTML',
)
popup_template_url = resource_download_url(html_dataset, popup_template_resource)

sites_df = sites_df.drop(columns=['popup_resource_id', 'popup_url'], errors='ignore').copy()
sites_df['popup_resource_id'] = popup_template_resource['id']
sites_df['popup_url'] = sites_df.apply(lambda row: build_popup_url(popup_template_url, row), axis=1)
sites_df[['name_condensed', 'measurement_url', 'popup_url']].head(2)


## Create the Folium Map
With a CKAN-hosted popup template available, assemble the interactive map with base layers, the CKAN-hosted OPERA subsidence COG overlay, extensometer markers, and county outlines. Then save the finished map HTML locally for upload.


In [ ]:
MAP_SIMPLIFY_TOLERANCE = float(os.environ.get('MAP_SIMPLIFY_TOLERANCE', '0.005'))
INCLUDE_COUNTY_BOUNDARIES = os.environ.get('INCLUDE_COUNTY_BOUNDARIES', '1') == '1'
OPERA_COG_DATASET_ID = 'houston-opera-subsidence-estimates'
OPERA_COG_RESOURCE_NAME = 'Opera_disp.tif'


def ckan_package_show(dataset_id):
    response = requests.get(f'{CKAN_URL}/api/3/action/package_show', params={'id': dataset_id}, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if not payload.get('success'):
        raise RuntimeError(payload.get('error') or payload)
    return payload['result']


def ckan_resource_url(dataset_id, resource_name):
    dataset = ckan_package_show(dataset_id)
    for resource in dataset.get('resources', []):
        if resource.get('name') == resource_name:
            return resource['url']
    raise ValueError(f"Resource {resource_name!r} not found in {dataset_id!r}")


class GeoTiffOverlay(MacroElement):
    def __init__(self, cog_url, layer_group, *, opacity=0.65):
        super().__init__()
        self._name = 'GeoTiffOverlay'
        self.cog_url = cog_url
        self.layer_group = layer_group
        self.opacity = opacity
        self._template = Template("""
            {% macro header(this, kwargs) %}
            <script src="https://unpkg.com/georaster"></script>
            <script src="https://unpkg.com/georaster-layer-for-leaflet"></script>
            <script src="https://unpkg.com/chroma-js@2.4.2/chroma.min.js"></script>
            {% endmacro %}
            {% macro script(this, kwargs) %}
            (function() {
                const cogUrl = {{ this.cog_url|tojson }};
                const layerGroup = {{ this.layer_group.get_name() }};
                const opacity = {{ this.opacity }};
                const colorScale = chroma.scale(['#2c7bb6', '#ffffbf', '#d7191c']);

                function finiteRasterRange(georaster) {
                    const raster = georaster.values[0];
                    const noDataValue = georaster.noDataValue;
                    let min = Infinity;
                    let max = -Infinity;
                    for (const row of raster) {
                        for (const value of row) {
                            if (Number.isFinite(value) && value !== noDataValue) {
                                min = Math.min(min, value);
                                max = Math.max(max, value);
                            }
                        }
                    }
                    if (!Number.isFinite(min) || !Number.isFinite(max)) {
                        return null;
                    }
                    if (min === max) {
                        min -= 1;
                        max += 1;
                    }
                    return { min, max };
                }

                fetch(cogUrl)
                    .then(function(response) {
                        if (!response.ok) {
                            throw new Error('HTTP ' + response.status);
                        }
                        return response.arrayBuffer();
                    })
                    .then(parseGeoraster)
                    .then(function(georaster) {
                        const range = finiteRasterRange(georaster);
                        if (!range) {
                            console.warn('OPERA COG layer has no finite raster values.');
                            return;
                        }
                        const noDataValue = georaster.noDataValue;
                        const layer = new GeoRasterLayer({
                            georaster: georaster,
                            opacity: opacity,
                            resolution: 128,
                            pixelValuesToColorFn: function(values) {
                                const value = values[0];
                                if (!Number.isFinite(value) || value === noDataValue) {
                                    return null;
                                }
                                const normalized = (value - range.min) / (range.max - range.min);
                                return colorScale(Math.max(0, Math.min(1, normalized))).hex();
                            }
                        });
                        layer.addTo(layerGroup);
                    })
                    .catch(function(error) {
                        console.error('Could not load OPERA COG layer:', error);
                    });
            })();
            {% endmacro %}
        """)


opera_cog_url = ckan_resource_url(OPERA_COG_DATASET_ID, OPERA_COG_RESOURCE_NAME)

m = folium.Map((29.7001, -95.3701), zoom_start=9, tiles="OpenStreetMap", name='Open Street Map')
folium.TileLayer("cartodb positron", name='Carto DB', show=False).add_to(m)

fg_opera = folium.FeatureGroup(name='OPERA subsidence estimates', show=False)
fg_opera.add_to(m)
GeoTiffOverlay(opera_cog_url, fg_opera, opacity=0.65).add_to(m)

fg_extensometers = folium.FeatureGroup(name='extensometers')

# Add compaction sites to map. The iframe source points to the reusable popup
# template with this marker's Upstream measurement endpoint in the query string.
for _, site_row in sites_df.iterrows():
    iframe_src = site_row['popup_url'].replace('&', '&amp;')
    html = f'<iframe src="{iframe_src}" width="540" height="380" frameborder="0" loading="lazy"></iframe>'
    popup = folium.Popup(folium.Html(html, script=True), max_width=570)
    folium.Marker(
        location=[site_row['DEC_LAT_VA'], site_row['DEC_LONG_VA']],
        tooltip=site_row['STATION_NM'],
        popup=popup,
    ).add_to(fg_extensometers)
fg_extensometers.add_to(m)

# Harris County Boundary
county_list = ['Harris', 'Fort Bend', 'Galveston']
if INCLUDE_COUNTY_BOUNDARIES:
    for county in county_list:
        county_geom = tx_gdf_county[tx_gdf_county['CNTY_NM'] == county].copy()
        county_geom.geometry = county_geom.geometry.simplify(MAP_SIMPLIFY_TOLERANCE, preserve_topology=True)
        feature_name = county + ' County'
        if county == 'Harris':
            show_choice = True
        else:
            show_choice = False
        folium.GeoJson(
            county_geom,
            name=feature_name,
            show=show_choice,
            style_function=lambda feature: {
                'fillOpacity': 0.1,
            },
        ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

map_output_path = output_dir / 'index.html'
m.save(map_output_path.as_posix())
map_size_mb = round(file_size_mb(map_output_path), 3)
map_output_path, map_size_mb


### Upload the Final Folium Map and Create a CKAN Web View
Upload the saved `index.html` map to CKAN, then create or update a `webpage_view` so the resource page can render the map inside CKAN via iframe.


In [ ]:
map_size_mb = file_size_mb(map_output_path)
if map_size_mb > MAX_UPLOAD_MB_WARNING:
    raise ValueError(
        f"index.html is {map_size_mb:.2f} MB. Increase MAP_SIMPLIFY_TOLERANCE or set INCLUDE_COUNTY_BOUNDARIES=0 before uploading."
    )

map_resource = upsert_resource_by_name(
    ckan_client,
    html_dataset,
    map_output_path,
    name='index.html',
    description='Folium overview map for the Houston-area extensometer compaction tutorial.',
    format_name='HTML',
)
map_resource_url = resource_download_url(html_dataset, map_resource)
map_view = upsert_webpage_view(
    ckan_client,
    map_resource['id'],
    title='Interactive Folium Map',
    description='Embedded CKAN web view for the uploaded Folium map resource.',
)
{
    'map_resource_url': map_resource_url,
    'resource_page_url': f"{CKAN_URL}/dataset/{html_dataset['name']}/resource/{map_resource['id']}",
    'resource_view_id': map_view['id'],
}
